# Calculating Power Law Distribution of Allan Factor

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = 'allan_var-difs.tsv'

PARAMETERS_PATH = 'av-params-difs.csv'

vis_path = DATA_PATH.replace('.csv', '.html')

In [3]:
subcorpora_col = 'corpus'

In [4]:
# to rename the corpus correctly . . . 
def lang(x):
    result = str(x)
    
    if '-DISPEL-' in result:
        result = 'DISPEL'
    
    if result.startswith('GCSAusE'):
        result = 'GCSAusE'
    
    if result.startswith('se'):
        result = 'CORAAL'
        
    if result.startswith('call'):
        result = x.split('-')[0]
    
    if result.startswith('MICASE'):
        result = 'MICASE'
    
    if result.startswith('instruction'):
        result = x.split('-')[1]
    
    if result.startswith('CABNC'):
        result = 'CABNC'
        
    if result.startswith('SBCSAE'):
        result = 'SBCSAE'
        
    if result.startswith('SCoSE'):
        result = 'SCoSE'
        
    if result.startswith('candor'):
        result = 'CANDOR'
    
    if result.startswith('grace'):
        result = 'Miao-fNIRS'
    
    return result

## Looking at Results

In [5]:
results = pd.read_csv(PARAMETERS_PATH)
results.shape

(496287, 6)

In [6]:
results.isna().sum()


x_turn_id          0
self               0
conversation_id    0
speaker            0
slope              3
b0                 1
dtype: int64

In [7]:
results.isna().mean()

x_turn_id          0.000000
self               0.000000
conversation_id    0.000000
speaker            0.000000
slope              0.000006
b0                 0.000002
dtype: float64

In [8]:
(results['slope'].abs() == np.inf).sum(), (results['slope'].abs() == np.inf).mean()

(np.int64(7), np.float64(1.4104741812701119e-05))

In [9]:
results.head()

,x_turn_id,self,conversation_id,speaker,slope,b0
0,CABNC-0missing-CABNC-0missing-KB0RE004-6,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-PS009,0.557254,-5.847243
1,CABNC-0missing-CABNC-0missing-KB0RE006-16,True,CABNC-0missing-CABNC-0missing-KB0RE006,CABNC-0missing-CABNC-0missing-KB0RE006-PS007,-1.022819,-0.764458
2,CABNC-0missing-CABNC-0missing-KB1RE005-10,True,CABNC-0missing-CABNC-0missing-KB1RE005,CABNC-0missing-CABNC-0missing-KB1RE005-PS01A,-1.545812,-7.791417
3,CABNC-0missing-CABNC-0missing-KB1RE005-100,True,CABNC-0missing-CABNC-0missing-KB1RE005,CABNC-0missing-CABNC-0missing-KB1RE005-PS01F,-0.949632,-7.287535
4,CABNC-0missing-CABNC-0missing-KB1RE005-101,True,CABNC-0missing-CABNC-0missing-KB1RE005,CABNC-0missing-CABNC-0missing-KB1RE005-PS01A,-0.695415,-6.930180


Adding Corpus names back in . . .

In [10]:
results[subcorpora_col] = [lang(x) for x in tqdm(results['conversation_id'])]

  0%|          | 0/496287 [00:00<?, ?it/s]

In [11]:
results[subcorpora_col].value_counts()

corpus
CABNC          143703
CORAAL         132608
MICASE         102568
callhome        34021
SBCSAE          30846
callfriend      16364
Miao-fNIRS      15379
CANDOR           8816
SCoSE            6394
GCSAusE          3115
DISPEL           1906
Graesser          396
med_school        136
Frederiksen        35
Name: count, dtype: int64

In [12]:
results.head()

,x_turn_id,self,conversation_id,speaker,slope,b0,corpus
0,CABNC-0missing-CABNC-0missing-KB0RE004-6,True,CABNC-0missing-CABNC-0missing-KB0RE004,CABNC-0missing-CABNC-0missing-KB0RE004-PS009,0.557254,-5.847243,CABNC
1,CABNC-0missing-CABNC-0missing-KB0RE006-16,True,CABNC-0missing-CABNC-0missing-KB0RE006,CABNC-0missing-CABNC-0missing-KB0RE006-PS007,-1.022819,-0.764458,CABNC
2,CABNC-0missing-CABNC-0missing-KB1RE005-10,True,CABNC-0missing-CABNC-0missing-KB1RE005,CABNC-0missing-CABNC-0missing-KB1RE005-PS01A,-1.545812,-7.791417,CABNC
3,CABNC-0missing-CABNC-0missing-KB1RE005-100,True,CABNC-0missing-CABNC-0missing-KB1RE005,CABNC-0missing-CABNC-0missing-KB1RE005-PS01F,-0.949632,-7.287535,CABNC
4,CABNC-0missing-CABNC-0missing-KB1RE005-101,True,CABNC-0missing-CABNC-0missing-KB1RE005,CABNC-0missing-CABNC-0missing-KB1RE005-PS01A,-0.695415,-6.930180,CABNC


In [13]:
sel_ = results['slope'].isna() | (results['slope'].abs() == np.inf) 
sel_.sum(), sel_.mean(), (~sel_).sum()

(np.int64(10), np.float64(2.01496311610016e-05), np.int64(496277))

In [14]:
sel_ = ~sel_

In [15]:
results['self'].value_counts()

self
True    496287
Name: count, dtype: int64

In [16]:
mu, SE = (results['slope'].loc[sel_].mean(), results['slope'].loc[sel_].std()/np.sqrt(sel_.sum()))
mu, SE

(np.float64(-0.1779769326432609), np.float64(0.0015493445733383055))

t-test for good measure.

In [17]:
mu / SE

np.float64(-114.8724019859454)

#### individual corpora statistics

In [18]:
subcorpus_results = []

In [19]:
for corpus in tqdm(results[subcorpora_col].unique()):
    
    res = {'corpus': corpus}
    
    sub = results.loc[
        results[subcorpora_col].isin([corpus])
        & sel_
    ]
    
    # self vals
    sel__ = sub['self']
    mu, se, n = sub['slope'].loc[sel__].mean(), sub['slope'].loc[sel__].std() / np.sqrt(sel__.sum()), sel__.sum()
    res['alpha'] = mu
    res['se'] = se
    res['n'] = n
    res['t'] = 't({})={}'.format(n-1, mu/se)
    
    # save outputs
    subcorpus_results += [res]

subcorpus_results = pd.DataFrame(subcorpus_results)    

  0%|          | 0/14 [00:00<?, ?it/s]

In [20]:
subcorpus_results.head(n=100)

,corpus,alpha,se,n,t
0,CABNC,-0.180358,0.003104,143698,t(143697)=-58.09954279242669
1,GCSAusE,-0.162202,0.022294,3115,t(3114)=-7.275654144218614
2,MICASE,-0.169334,0.003387,102563,t(102562)=-49.99167186911027
3,SBCSAE,-0.198740,0.006305,30846,t(30845)=-31.522368876220863
4,SCoSE,-0.199344,0.013537,6394,t(6393)=-14.725427549904927
5,callfriend,-0.162194,0.008065,16364,t(16363)=-20.10977930390236
6,callhome,-0.173074,0.006160,34021,t(34020)=-28.094687519471503
7,CANDOR,-0.087548,0.011699,8816,t(8815)=-7.483711690731307
8,Miao-fNIRS,-0.170991,0.008213,15379,t(15378)=-20.819079210988036
9,DISPEL,-0.123257,0.031731,1906,t(1905)=-3.8844744490379073


In [21]:
# reporting['Var'] = reporting.index.values
# 
# with open(lme_results_name.replace('.csv', '2.txt'), 'w') as f:
#     txt =  reporting[['Var', 'coefs', 'stat', 'p']].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')
# 
#     txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
#     f.write(txt)
#     f.close()

## Visualization

In [22]:
import plotly.express as px
import plotly.graph_objects as go

In [23]:
df = pd.read_table(DATA_PATH, sep='\t')

In [24]:
df[subcorpora_col] = [lang(x) for x in tqdm(df['conversation_id'])]

  0%|          | 0/8513321 [00:00<?, ?it/s]

In [25]:
df.head()

,nx,ny,self,groups,x_speaker,dyad,conversation_id,turn_delta,sample_delta,residual,resid,resid_1,resid_2,N,tau,allan_var,corpus
0,20.0,12.0,True,GCSAusE-GCSAusE-01-6,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01,3,1,-0.555195,0.984138,-0.217703,-0.422610,26,26,0.000735,GCSAusE
1,20.0,5.0,True,GCSAusE-GCSAusE-01-6,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01,5,2,0.213904,-0.217703,-0.422610,0.248789,26,26,0.000568,GCSAusE
2,20.0,7.0,True,GCSAusE-GCSAusE-01-6,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01,8,3,-0.034928,-0.422610,0.248789,0.319591,26,26,0.000267,GCSAusE
3,20.0,8.0,True,GCSAusE-GCSAusE-01-6,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01,14,4,-0.157208,0.248789,0.319591,-0.527367,26,26,0.000623,GCSAusE
4,20.0,13.0,True,GCSAusE-GCSAusE-01-6,GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01-S-GCSAusE-GCSAusE-01-S,GCSAusE-GCSAusE-01,20,5,-0.024097,0.319591,-0.527367,-0.597210,26,26,0.000447,GCSAusE


In [26]:
group_columns = ['sample_delta']
value_columns = ['resid']

In [27]:
df_ = df[group_columns+value_columns].groupby(by=group_columns).agg('mean').reset_index()
df_['std'] = df[group_columns+value_columns].groupby(by=group_columns).agg('std').reset_index()[value_columns[0]]
df_['Nk'] = df[group_columns+value_columns].groupby(by=group_columns).agg('count').reset_index()[value_columns[0]]
df_['SE'] = df_['std'] / np.sqrt(df_['Nk'])

df_.shape

(87, 5)

In [28]:
df_ = df_.rename(columns={'resid': 'Δ Hxy'})

In [29]:
df_.head()

,sample_delta,Δ Hxy,std,Nk,SE
0,1,0.174221,0.910559,514886,0.001269
1,2,0.135934,0.831381,505500,0.001169
2,3,0.109882,0.816311,496287,0.001159
3,4,0.096046,0.791889,487390,0.001134
4,5,0.089813,0.793147,478424,0.001147


In [30]:
df_=df_.loc[(df_['sample_delta'] <= 30)]

In [31]:
new_fig = go.Figure()

In [32]:
line_color_code = 'rgba(65, 105, 225, 1.0)'
ribbon_color_code = 'rgba(65, 105, 225, 0.2)'

In [33]:
new_fig.add_trace(
    go.Scatter(
        x=df_['sample_delta'].values,
        y=df_['Δ Hxy'].values,
        mode='markers+lines',
        name='Δ Hxy',
        legendgroup='Δ Hxy',
        showlegend=False,
        line=dict(color=line_color_code)
    )
)

# upper bounds for ribbon
new_fig.add_trace(
    go.Scatter(
        x=df_['sample_delta'].values,
        y=df_['Δ Hxy'].values + (2*df_['SE'].values),
        mode='lines',
        line=dict(color=line_color_code,width=0),
        name='Δ Hxy'+'_upper_bound',
        showlegend=False,
        legendgroup='Δ Hxy',
        fill='tonexty',
        fillcolor=ribbon_color_code,
    ), 
    # row=row, col=col
)

# lower bounds for ribbon
new_fig.add_trace(
    go.Scatter(
        x=df_['sample_delta'].values,
        y=df_['Δ Hxy'].values - (2*df_['SE'].values),
        mode='lines',
        line=dict(color=line_color_code,width=0),
        name='Δ Hxy'+'_lower_bound',
        showlegend=False,
        legendgroup='Δ Hxy',
        fill='tonexty',
        fillcolor=ribbon_color_code,
    ), 
    # row=row, col=col
)

new_fig.add_hline(y=0,line_color='darkgray',line_width=2)

In [34]:
new_fig.update_layout(
    template='plotly_white',
    xaxis=dict(
        title='turn Δ'
    ),
    yaxis=dict(
        title='Δ Hxy (other - self)'
    ),
    margin=dict(
        l=0,
        r=0,
        b=15,
        t=15
    )
)
new_fig.show()

In [35]:
# new_fig.write_html('delta-Hxy.html')

### Individual corpora

##### grouping data

In [36]:
group_columns = [subcorpora_col, 'sample_delta']
value_columns = ['resid']

In [37]:
df_ = df[group_columns+value_columns].groupby(by=group_columns).agg('mean').reset_index()
df_['std'] = df[group_columns+value_columns].groupby(by=group_columns).agg('std').reset_index()[value_columns[0]]
df_['Nk'] = df[group_columns+value_columns].groupby(by=group_columns).agg('count').reset_index()[value_columns[0]]
df_['SE'] = df_['std'] / np.sqrt(df_['Nk'])

df_.shape

(406, 6)

In [38]:
df_ = df_.rename(columns={'resid': 'Δ Hxy'})

In [39]:
df_.head()

,corpus,sample_delta,Δ Hxy,std,Nk,SE
0,CABNC,1,0.110920,0.743805,152097,0.001907
1,CABNC,2,0.085467,0.636622,147749,0.001656
2,CABNC,3,0.073043,0.623445,143703,0.001645
3,CABNC,4,0.059874,0.557559,139776,0.001491
4,CABNC,5,0.054814,0.556331,136173,0.001508


In [40]:
df_=df_.loc[(df_['sample_delta'] <= 20)]

##### making visualization

In [41]:
line_color_code = 'rgba(65, 105, 225, 1.0)'
ribbon_color_code = 'rgba(65, 105, 225, 0.2)'

In [42]:
if not os.path.exists('individual_corpora'):
    os.mkdir('individual_corpora')

In [43]:
for corpus in tqdm(df_[subcorpora_col].unique()):
    sub = df_.loc[df_[subcorpora_col].isin([corpus])]
    
    new_fig = go.Figure()
    
    new_fig.add_trace(
        go.Scatter(
            x=sub['sample_delta'].values,
            y=sub['Δ Hxy'].values,
            mode='markers+lines',
            name='Δ Hxy',
            legendgroup='Δ Hxy',
            showlegend=False,
            line=dict(color=line_color_code)
        )
    )
    
    # upper bounds for ribbon
    new_fig.add_trace(
        go.Scatter(
            x=sub['sample_delta'].values,
            y=sub['Δ Hxy'].values + (2*sub['SE'].values),
            mode='lines',
            line=dict(color=line_color_code,width=0),
            name='Δ Hxy'+'_upper_bound',
            showlegend=False,
            legendgroup='Δ Hxy',
            fill='tonexty',
            fillcolor=ribbon_color_code,
        ), 
        # row=row, col=col
    )
    
    # lower bounds for ribbon
    new_fig.add_trace(
        go.Scatter(
            x=sub['sample_delta'].values,
            y=sub['Δ Hxy'].values - (2*sub['SE'].values),
            mode='lines',
            line=dict(color=line_color_code,width=0),
            name='Δ Hxy'+'_lower_bound',
            showlegend=False,
            legendgroup='Δ Hxy',
            fill='tonexty',
            fillcolor=ribbon_color_code,
        ), 
        # row=row, col=col
    )
    
    new_fig.add_hline(y=0,line_color='darkgray',line_width=2)
    
    new_fig.update_layout(
        template='plotly_white',
        xaxis=dict(
            title='turn Δ'
        ),
        yaxis=dict(
            title='Δ Hxy (other - self)'
        ),
        margin=dict(
            l=0,
            r=0,
            b=15,
            t=15
        )
    )
    
    new_fig.write_html(
        os.path.join('individual_corpora', corpus + '.html')
    )

  0%|          | 0/14 [00:00<?, ?it/s]

### Add figure to subplots instead.

In [67]:
from plotly.subplots import make_subplots
from plotly.offline import plot as img_out

In [73]:
corpora = df_[subcorpora_col].unique()

In [74]:
k_cols = 2
k_rows = int(np.ceil(len(corpora)/k_cols))

In [75]:
main_fig = make_subplots(
    rows=k_rows,cols=k_cols,
    subplot_titles=corpora
)

In [76]:
positions = [
    [(row+1, col+1) for col in range(k_cols)] for row in range(k_rows)
]
positions = sum(positions,[])

In [77]:
ct = 0
for corpus in tqdm(corpora):
    sub = df_.loc[df_[subcorpora_col].isin([corpus])]

    row, col = positions[ct]
    print(row,col)
    main_fig.add_trace(
        go.Scatter(
            x=sub['sample_delta'].values,
            y=sub['Δ Hxy'].values,
            mode='markers+lines',
            name='Δ Hxy',
            legendgroup='Δ Hxy',
            showlegend=False,
            line=dict(color=line_color_code)
        ),
        row=row,col=col
    )

    # upper bounds for ribbon
    main_fig.add_trace(
        go.Scatter(
            x=sub['sample_delta'].values,
            y=sub['Δ Hxy'].values + (2*sub['SE'].values),
            mode='lines',
            line=dict(color=line_color_code,width=0),
            name='Δ Hxy'+'_upper_bound',
            showlegend=False,
            legendgroup='Δ Hxy',
            fill='tonexty',
            fillcolor=ribbon_color_code,
        ),
        row=row, col=col
    )

    # lower bounds for ribbon
    main_fig.add_trace(
        go.Scatter(
            x=sub['sample_delta'].values,
            y=sub['Δ Hxy'].values - (2*sub['SE'].values),
            mode='lines',
            line=dict(color=line_color_code,width=0),
            name='Δ Hxy'+'_lower_bound',
            showlegend=False,
            legendgroup='Δ Hxy',
            fill='tonexty',
            fillcolor=ribbon_color_code,
        ),
        row=row, col=col
    )

    main_fig.add_hline(y=0,line_color='darkgray',line_width=2,row=row,col=col)

    ct+=1

  0%|          | 0/14 [00:00<?, ?it/s]

1 1
1 2
2 1
2 2
3 1
3 2
4 1
4 2
5 1
5 2
6 1
6 2
7 1
7 2


In [78]:
main_fig.update_layout(template='plotly_white')
# main_fig.update_layout(template='plotly_white',height=750,width=500)
main_fig.show()

In [79]:
main_fig.write_html('individual-plots.html')

In [80]:
# main_fig.to_image('individual-plots',format='svg')
main_fig.update_layout(template='plotly_white',height=750,width=500)
img_out(main_fig, image_filename='individual-plots.svg', image='svg')

'temp-plot.html'